# Lift entomológico — o vetor (MI-Aedes) ajuda a prever casos?

Esta é **a pergunta central da pesquisa**: o dado de armadilhas adiciona poder preditivo **além do clima**? Para responder de forma limpa, comparamos três modelos no **mesmo walk-forward multi-horizonte** (1–12 semanas):

- **só-clima** — replica o espírito do Oliveira 2025 (clima, sem armadilha);
- **clima + vetor** — adiciona o MI-Aedes;
- **só-vetor** — usa o vetor sem o clima.

Todos partem do **mesmo núcleo** (lags de casos + sazonalidade), então a diferença entre eles é **só o bloco de features** que muda. O *lift* = o quanto `clima+vetor` melhora sobre `só-clima`.

## 1. Carregar e montar as features

Mesma base e mesmas features de lag do `modelagem.ipynb`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def achar_raiz(marcador="Raspagem"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marcador).is_dir():
            return p
    raise FileNotFoundError(f"Pasta-raiz contendo '{marcador}/' nao encontrada a partir de {Path.cwd()}")


RAIZ = achar_raiz()
df = pd.read_csv(RAIZ / "Bases de dados" / "tabela_modelagem" / "tabela_final.csv",
                 parse_dates=["data_inicio_semana_epidemi"])
df = df.sort_values(["fonte", "data_inicio_semana_epidemi"]).reset_index(drop=True)
ALVO = "casos_confirmados"
g = df.groupby("fonte", group_keys=False)

for col in ["casos_confirmados", "aedes_aegypti_por_armadilha",
            "temp_media", "precip_total_mm", "orvalho_media", "umid_media", "pressao_media"]:
    for k in [1, 2, 3, 4]:
        df[f"{col}_lag{k}"] = g[col].shift(k)
df["casos_mm4"] = g["casos_confirmados"].transform(lambda s: s.rolling(4).mean())
df["vetor_mm4"] = g["aedes_aegypti_por_armadilha"].transform(lambda s: s.rolling(4).mean())
df["sem_sin"] = np.sin(2 * np.pi * df["semana"] / 52)
df["sem_cos"] = np.cos(2 * np.pi * df["semana"] / 52)

print("tabela:", df.shape)

## 2. Os três conjuntos de features

Classificamos cada feature em **núcleo** (casos + sazonalidade — sempre presente), **clima** (temperatura, chuva, umidade, pressão, radiação, vento, ENSO) e **vetor** (espécies, IMFA, nº de armadilhas). Os três modelos são o núcleo + combinações desses blocos.

In [ ]:
ignorar = ["fonte", "SE", "data_inicio_semana_epidemi", "ano", "semana", "interpolado"]
todas = [c for c in df.columns if c not in ignorar]

PADROES_VETOR = ("aedes", "culex", "armadilha", "vetor")
PADROES_CLIMA = ("temp", "precip", "orvalho", "umid", "pressao", "radiacao", "vento", "dias_de_chuva", "nino34", "oni")

vetor = [c for c in todas if any(t in c for t in PADROES_VETOR)]
clima = [c for c in todas if any(t in c for t in PADROES_CLIMA)]
nucleo = [c for c in todas if c not in vetor and c not in clima]   # casos (lags) + médias + sazonalidade

CONJUNTOS = {
    "so_clima": nucleo + clima,
    "clima_vetor": nucleo + clima + vetor,
    "so_vetor": nucleo + vetor,
}
print(f"núcleo: {len(nucleo)} | clima: {len(clima)} | vetor: {len(vetor)}")
print("núcleo:", nucleo)
print("vetor:", vetor)

## 3. Função: walk-forward multi-horizonte para um conjunto de features

Mesma estratégia direta do `modelagem.ipynb`, mas recebendo **qual conjunto de features** usar.

In [ ]:
from lightgbm import LGBMRegressor

PARAMS = dict(n_estimators=250, learning_rate=0.05, num_leaves=15,
              min_child_samples=5, verbose=-1, n_jobs=-1)


def walk_forward_conjunto(df, cols, alvo="casos_confirmados",
                          horizontes=range(1, 13), min_treino=104, passo=2):
    g = df.groupby("fonte", group_keys=False)
    linhas = []
    for h in horizontes:
        d = df.copy()
        d["y_h"] = g[alvo].shift(-h)
        sa = g["semana"].shift(-h)
        d["alvo_sin"] = np.sin(2 * np.pi * sa / 52)
        d["alvo_cos"] = np.cos(2 * np.pi * sa / 52)
        feats = cols + ["alvo_sin", "alvo_cos"]
        dh = d.dropna(subset=feats + ["y_h"]).sort_values("data_inicio_semana_epidemi").reset_index(drop=True)
        for i in range(min_treino, len(dh), passo):
            tr, te = dh.iloc[:i], dh.iloc[i:i + 1]
            m = LGBMRegressor(**PARAMS).fit(tr[feats], tr["y_h"])
            linhas.append({"h": h, "real": te["y_h"].values[0], "pred": m.predict(te[feats])[0]})
    return pd.DataFrame(linhas)


print("função pronta")

## 4. Rodar os três modelos

⏱️ 3 conjuntos × 12 horizontes (com `passo=2`) → **~15 minutos**. Aumente `passo` para iterar mais rápido.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

linhas = []
for nome, cols in CONJUNTOS.items():
    print("rodando:", nome, f"({len(cols)} features)")
    res = walk_forward_conjunto(df, cols)
    for h, x in res.groupby("h"):
        linhas.append({"conjunto": nome, "h": h,
                       "MAE": mean_absolute_error(x["real"], x["pred"]),
                       "R2": r2_score(x["real"], x["pred"])})
comp = pd.DataFrame(linhas)
print("pronto")

## 5. Comparação por horizonte + lift do vetor

In [ ]:
mae = comp.pivot(index="h", columns="conjunto", values="MAE").round(1)
r2 = comp.pivot(index="h", columns="conjunto", values="R2").round(3)
mae["lift_vetor_%"] = ((mae["so_clima"] - mae["clima_vetor"]) / mae["so_clima"] * 100).round(1)
print("=== MAE por horizonte ==="); print(mae.to_string())
print("\n=== R² por horizonte ==="); print(r2.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for c in ["so_clima", "clima_vetor", "so_vetor"]:
    ax[0].plot(mae.index, mae[c], "o-", label=c)
ax[0].set_title("MAE × horizonte"); ax[0].set_xlabel("semanas à frente"); ax[0].set_ylabel("MAE"); ax[0].legend(); ax[0].grid(alpha=0.3)
cores = ["tab:green" if v > 0 else "tab:red" for v in mae["lift_vetor_%"]]
ax[1].bar(mae.index, mae["lift_vetor_%"], color=cores)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("Lift do vetor (% redução do MAE de clima+vetor vs só-clima)")
ax[1].set_xlabel("semanas à frente"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Leitura honesta dos resultados

O resultado **não** é o simples "adicionar o vetor melhora tudo" — é mais interessante:

1. **Empilhar `clima+vetor` quase sempre PIORA vs `só-clima`** (lift negativo na maioria dos horizontes). Isso **não** significa que o vetor é inútil — significa **overfitting**: são ~54 features (44 só de clima, com muita redundância min/méd/máx) para apenas ~260 semanas. Jogar tudo junto adiciona ruído.

2. **Mas o `só-vetor` (10 features) empata ou SUPERA o `só-clima` (44 features)** em vários horizontes (h=1,2,3,6,7,8). Ou seja: o vetor atinge skill **comparável ou melhor com 4× menos variáveis**. **É a evidência de que o dado entomológico carrega sinal real e parcimonioso** — o coração da contribuição da tese.

3. **Nenhum modelo domina em todos os horizontes** — é um quadro misto, típico de série curta.

**Conclusão:** o caminho **não** é "jogar todas as features juntas", e sim **seleção de features (parcimônia)** — escolher as poucas variáveis de clima E de vetor que realmente importam e combiná-las. É aí que o lift verdadeiro deve aparecer.

## 7. Próximos passos

- **Seleção de features:** reduzir o bloco de clima (44→poucas) via importância/SHAP, e montar um `clima_enxuto + vetor` — testar se aí o lift fica positivo e consistente.
- **SHAP por horizonte** para ver quais features de vetor sustentam o `só-vetor`.
- **Regularizar** o LightGBM (mais `min_child_samples`, menos `num_leaves`, `reg_lambda`) — combate o overfitting com muitas features.
- Repetir com o **alvo = vetor** (PEP) e, se vier dado por bairro, no nível espacial.